# CTOAi Fine-Tuning: qwen2.5-coder:7b na OTClient/Tibia
**Google Colab T4** (darmowy) — LoRA fine-tuning z Unsloth

Czas treningu: ~30-60min na 5000 przykładach na T4 GPU

In [ ]:
# Krok 1: Instalacja Unsloth
%%capture
!pip install unsloth[colab-new] -q
!pip install --no-deps trl peft accelerate bitsandbytes -q

In [ ]:
# Krok 2: Upload datasetu (z lokalnego repo)
from google.colab import files
uploaded = files.upload()  # wgraj dataset.jsonl z training/data/
print('Plik wgrany:', list(uploaded.keys()))

In [ ]:
# Krok 3: Zaladuj model z 4-bit quantization
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-Coder-7B-Instruct',
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)
print('Model zaladowany OK')

In [ ]:
# Krok 4: LoRA adaptery
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('LoRA dodane. Trainable params:', model.print_trainable_parameters())

In [ ]:
# Krok 5: Wczytaj dataset
import json
from datasets import Dataset

dataset_file = list(uploaded.keys())[0]
examples = []
with open(dataset_file) as f:
    for line in f:
        examples.append(json.loads(line))

def format_messages(row):
    msgs = row['messages']
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    return {'text': text}

ds = Dataset.from_list(examples).map(format_messages)
print(f'Dataset: {len(ds)} przykladow')
print(ds[0]['text'][:300])

In [ ]:
# Krok 6: Trening
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    dataset_text_field='text',
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='linear',
        seed=42,
        output_dir='./ctoa-checkpoints',
    ),
)
print('Startuje trening...')
trainer.train()

In [ ]:
# Krok 7: Eksport do GGUF (dla Ollama)
model.save_pretrained_gguf(
    'ctoa-tibia-7b',
    tokenizer,
    quantization_method='q4_k_m',
)
print('Eksport GGUF gotowy!')
import os
gguf_file = [f for f in os.listdir('ctoa-tibia-7b') if f.endswith('.gguf')][0]
print(f'Plik: ctoa-tibia-7b/{gguf_file}')

In [ ]:
# Krok 8: Pobierz model
import shutil
shutil.make_archive('ctoa-tibia-7b-gguf', 'zip', 'ctoa-tibia-7b')
files.download('ctoa-tibia-7b-gguf.zip')
print('Pobieranie started. Nastepnie: wgraj .gguf na VPS i ollama create ctoa-tibia')

## Po pobraniu - wdrozenie na VPS

```bash
# Na VPS:
scp ctoa-tibia-7b.gguf ctoa-vps:/opt/ollama/models/

# Stworz Modelfile
cat > /tmp/Modelfile.tibia << EOF
FROM /opt/ollama/models/ctoa-tibia-7b.gguf
PARAMETER num_ctx 4096
PARAMETER temperature 0.3
EOF

ollama create ctoa-tibia -f /tmp/Modelfile.tibia

# Zaktualizuj kontener API
docker rm -f ctoa-api
docker run -d --name ctoa-api ... -e CTOA_LOCAL_MODEL_NAME=ctoa-tibia ...
```